<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_37_Cloud_Audit_Log_Anomalous_Download_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================================
# Experiment 9: Cloud Audit Log Anomalous Download Detector
# ==========================================================

from datetime import datetime
from collections import Counter

# Timestamp format
LOG_FMT = "%Y-%m-%d %H:%M:%S"


# ----------------------------------------------------------
# Function to build baseline IP addresses
# ----------------------------------------------------------
def build_baseline_ips(logs):
    """
    Build a baseline IP for each user based on the
    most frequently used IP address.
    """

    by_user = {}

    for entry in logs:
        by_user.setdefault(entry["user"], []).append(entry["ip"])

    baseline = {
        user: Counter(ips).most_common(1)[0][0]
        for user, ips in by_user.items()
    }

    return baseline


# ----------------------------------------------------------
# Function to detect anomalous downloads
# ----------------------------------------------------------
def flag_anomalous_downloads(
    logs,
    business_start=8,
    business_end=20
):
    """
    Flag download events that:
    1. Originate from an unfamiliar IP.
    2. Occur outside business hours.
    """

    baseline = build_baseline_ips(logs)

    flagged = []

    for entry in logs:

        if entry["action"] != "download":
            continue

        ts = datetime.strptime(
            entry["timestamp"],
            LOG_FMT
        )

        reasons = []

        if entry["ip"] != baseline.get(entry["user"]):
            reasons.append("IP differs from user baseline")

        if not (business_start <= ts.hour < business_end):
            reasons.append("Outside business hours")

        if reasons:
            flagged.append({
                **entry,
                "reasons": reasons
            })

    return flagged


# ==========================================================
# Test Cases
# ==========================================================

def test_experiment9():

    logs = [

        {
            "user": "alice",
            "action": "view",
            "file": "roadmap.docx",
            "timestamp": "2026-03-01 10:00:00",
            "ip": "10.0.0.5"
        },

        {
            "user": "alice",
            "action": "view",
            "file": "budget.xlsx",
            "timestamp": "2026-03-02 11:00:00",
            "ip": "10.0.0.5"
        },

        {
            "user": "alice",
            "action": "download",
            "file": "report.pdf",
            "timestamp": "2026-03-03 14:00:00",
            "ip": "10.0.0.5"
        },

        {
            "user": "alice",
            "action": "download",
            "file": "customer_database.csv",
            "timestamp": "2026-03-05 02:15:00",
            "ip": "185.220.101.7"
        }

    ]

    # Build baseline IPs
    baseline = build_baseline_ips(logs)

    print("User Baseline IPs")
    print("-" * 50)

    for user, ip in baseline.items():
        print(f"{user} : {ip}")

    print()

    # Detect anomalies
    flagged = flag_anomalous_downloads(logs)

    print("Anomalous Download Detection")
    print("-" * 50)

    for event in flagged:
        print(f"User      : {event['user']}")
        print(f"File      : {event['file']}")
        print(f"Time      : {event['timestamp']}")
        print(f"IP        : {event['ip']}")
        print("Reasons   :")

        for reason in event["reasons"]:
            print(f" - {reason}")

        print()

    # Assertions
    assert len(flagged) == 1
    assert flagged[0]["file"] == "customer_database.csv"
    assert "IP differs from user baseline" in flagged[0]["reasons"]
    assert "Outside business hours" in flagged[0]["reasons"]

    print("All test cases passed.")


# ----------------------------------------------------------
# Run Test
# ----------------------------------------------------------

test_experiment9()

User Baseline IPs
--------------------------------------------------
alice : 10.0.0.5

Anomalous Download Detection
--------------------------------------------------
User      : alice
File      : customer_database.csv
Time      : 2026-03-05 02:15:00
IP        : 185.220.101.7
Reasons   :
 - IP differs from user baseline
 - Outside business hours

All test cases passed.
